# 01 Dataset Audit

Purpose:
- validate the raw dataset structure
- count participants and recordings by class
- detect missing files, inconsistent naming, and leakage risks

Primary questions:
- How many participants exist in each class?
- How many recordings exist per participant?
- Are there missing or duplicate files?
- Can we reliably identify each patient for patient-level modeling?


In [ ]:
from pathlib import Path

def find_data_root():
    cwd = Path.cwd().resolve()
    # prefer a repository-level data/ if present, otherwise prefer backend/data
    for path in [cwd, *cwd.parents]:
        if (path / 'data').exists():
            return path
    return cwd

PROJECT_ROOT = find_data_root()
# common data layouts this project uses: repo-root/data or backend/data
POSSIBLE_DATA_DIRS = [PROJECT_ROOT / 'data', PROJECT_ROOT / 'backend' / 'data']
DATA_DIR = next((d for d in POSSIBLE_DATA_DIRS if d.exists()), PROJECT_ROOT / 'data')
print('Using data dir:', DATA_DIR)
DATA_DIR

In [ ]:
# Dataset audit: count subjects and recordings
from pathlib import Path
try:
    import pandas as pd
except Exception:
    pd = None

# Try common dataset folder name used in this repo
RAW_DIR = DATA_DIR / 'audio_lanzhou_2015' if (DATA_DIR / 'audio_lanzhou_2015').exists() else DATA_DIR / 'raw'
print('Looking for raw data in', RAW_DIR)
if not RAW_DIR.exists():
    print('Raw data directory not found:', RAW_DIR)
else:
    subjects = []
    for subj in sorted([p for p in RAW_DIR.iterdir() if p.is_dir()]):
        wavs = list(subj.rglob('*.wav'))
        subjects.append({'subject': subj.name, 'n_recordings': len(wavs), 'path': str(subj)})
    if pd is not None:
        df = pd.DataFrame(subjects).sort_values('subject')
        display(df)
        print('Total subjects:', len(df))
        print('Total recordings:', int(df['n_recordings'].sum()))
    else:
        print('pandas not available, printing summary:')
        total_subjects = len(subjects)
        total_recordings = sum(s['n_recordings'] for s in subjects)
        print('Total subjects:', total_subjects)
        print('Total recordings:', total_recordings)

# Attempt to locate subject metadata workbook
META_LOCATIONS = [DATA_DIR / 'subjects_information_audio_lanzhou_2015.xlsx', PROJECT_ROOT / 'subjects_information_audio_lanzhou_2015.xlsx']
meta_path = next((p for p in META_LOCATIONS if p.exists()), None)
if meta_path:
    print('Found metadata workbook at', meta_path)
    if pd is not None:
        try:
            meta = pd.read_excel(meta_path)
            display(meta.head())
            print('Metadata rows:', len(meta))
        except Exception as e:
            print('Failed to read metadata workbook:', e)
else:
    print('No metadata workbook found at expected locations')